In [ ]:
%matplotlib widget
%reset -f
%load_ext autoreload
%autoreload 1
%aimport mechanics

from sympy import *
from mechanics import *
import mechanics.config

mechanics.config.diff_notations = {}

BC_0 = 'free' # 'free' or 'fixed'
BC_L = 'fixed'  # 'free' or 'fixed'

t, x = base_spaces('t x')

u, = variables('u', t, x)
c, L = constants('c L')
    
eq = {diff(u, t, t): c**2 * diff(u, x, x)}
show_equations(eq)

bc = {
    (diff(u, x) if BC_0 == 'free' else u).subs(x, 0): 0,
    (diff(u, x) if BC_L == 'free' else u).subs(x, L): 0,
}
show_equations(bc)

In [ ]:
from mechanics.difference import second_central, central
from mechanics.space import Z

i, j = indices('i j')
h, k = constants('h k')
N, M = constants('N M', space=Z)

d = discretization()\
    .space(t, i, h).diff(t, second_central)\
    .space(x, j, k).diff(x, second_central).diff(x, central)

u = d(u)

eq_d = d(eq)
show_equations(eq_d)

eq_d = solve(to_implicit(eq_d), u[i+1,j])
show_equations(eq_d)

bc_d = d(bc)
show_equations(bc_d)

bc_solve_for = (
    u[:,-1] if BC_0 == 'free' else u[:,0],
    u[:,M+1] if BC_L == 'free' else u[:,M]
)
bc_d = solve((f.subs(L/k, M) for f in to_implicit(bc_d)), bc_solve_for)
show_equations(bc_d)

In [ ]:
%autoreload

j_min = -1 if BC_0 == 'free' else 0
j_max = M+1 if BC_L == 'free' else M

solver = build_solver()
solver.constants(c, h, k, N, M)
solver.variables(u, indices=((i, -1, N), (j, j_min, j_max)))
solver.inputs(u[0,:], u[-1,:])
with solver.steps(i, 0, N-1) as step_time:
    step_time.explicit(bc_d)
    with step_time.steps(j, j_min+1, j_max-1) as step_space:
        step_space.explicit(eq_d)
solver = solver.generate()

In [ ]:
%autoreload
import numpy as np

_ = solver.run({
    c: 1.0, 
    h: 0.01, k: 0.01, 
    N: 4 / h, M: 1 / k,
    u[0,:]:  1 - tanh((j * k) * 5)**2,
    u[-1,:]: 1 - tanh((j * k) * 5)**2,
    # u[0,:]:  sin(j * k * np.pi),
    # u[-1,:]:  sin(j * k * np.pi),
})

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

# t = np.arange(0, _[T], _[h])

fig, ax = plt.subplots(figsize=(4, 4), subplot_kw={"projection": "3d"}, tight_layout=True)
T = np.arange(0, _[u].shape[0]) * _[h]
X = np.arange(0, _[u].shape[1]) * _[k]
T, X = np.meshgrid(T, X, indexing='ij')
ax.plot_surface(T, X, _[u], cmap='coolwarm')
ax.set_ylabel('$x$')
ax.set_zlabel('$u$')
plt.show()
